In [14]:
import joblib
import pandas as pd

from sklearn.metrics import f1_score
from sklearn.base import BaseEstimator, TransformerMixin

In [17]:
df = pd.read_csv('parkinsons.data')

В этом ноутбуке я привела код, который нужен чтобы считать наши модели и запустить :)

Тестирую запуск на адекватность на исходных данных

In [8]:
# test log reg

cols_to_drop = ['MDVP:RAP','MDVP:PPQ','Jitter:DDP', 'MDVP:Shimmer(dB)',
                'Shimmer:APQ3', 'Shimmer:APQ5', 'MDVP:APQ', 'Shimmer:DDA',
                'PPE']

X_testik = df.drop(columns=['name', 'status']+cols_to_drop)
y_testik = df['status']

log_pipe = joblib.load('log_reg_pipline.joblib')

preds = log_pipe.predict(X_testik)
print(f"Точность: {f1_score(preds, y_testik):.3f}")

Точность: 0.918


In [9]:
# test knn
knn_scaler = joblib.load('knn_scaler.joblib')
knn_model = joblib.load('knn_model.joblib')

X_testik = df.drop(columns=['name', 'status'])
y_testik = df['status']

X_testik = knn_scaler.transform(X_testik)

preds = knn_model.predict(X_testik)
print(f"Точность: {f1_score(preds, y_testik):.3f}")

Точность: 0.953


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RobustScaler was fitted without feature names
  warnings.warn(


In [10]:
# test svm
svm_scaler = joblib.load('svm_scaler.joblib')
svm_model = joblib.load('svm_model.joblib')

X_testik = df.drop(columns=['name', 'status'])
y_testik = df['status']

X_testik = svm_scaler.transform(X_testik)

preds = svm_model.predict(X_testik)
print(f"Точность: {f1_score(preds, y_testik):.3f}")

Точность: 0.918


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RobustScaler was fitted without feature names
  warnings.warn(


In [11]:
# test dec tree
dt_model = joblib.load('dt_model.joblib')

X_testik = df.drop(columns=['name', 'status'])
y_testik = df['status']

preds = dt_model.predict(X_testik)
print(f"Точность: {f1_score(preds, y_testik):.3f}")

Точность: 0.877


In [12]:
# test random forest
rf_model = joblib.load('rf_model.joblib')

X_testik = df.drop(columns=['name', 'status'])
y_testik = df['status']

preds = rf_model.predict(X_testik)
print(f"Точность: {f1_score(preds, y_testik):.3f}")

Точность: 0.997


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


In [15]:
# test xgb

class IQRCapper(BaseEstimator, TransformerMixin):
    def __init__(self, factor=1.5):
        self.factor = factor
        self.lower_bounds_ = None
        self.upper_bounds_ = None
        self.columns_ = None

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.columns_ = X.columns
        q1 = X.quantile(0.25)
        q3 = X.quantile(0.75)
        iqr = q3 - q1
        self.lower_bounds_ = q1 - self.factor * iqr
        self.upper_bounds_ = q3 + self.factor * iqr
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        X.columns = self.columns_
        for col in self.columns_:
          X[col] = X[col].clip(self.lower_bounds_[col], self.upper_bounds_[col])
          return X


xgb_model = joblib.load('xgb_model.joblib')

X_testik = df.drop(columns=['name', 'status'])
y_testik = df['status']

preds = xgb_model.predict(X_testik)
print(f"Точность: {f1_score(preds, y_testik):.3f}")

Точность: 0.948
